In [1]:

import numpy as np
import os
# from xacrodoc import XacroDoc
from pydrake.common import temp_directory
from pydrake.geometry import SceneGraphConfig, StartMeshcat, SceneGraph
from pydrake.math import RigidTransform, RollPitchYaw, RotationMatrix
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.systems.analysis import Simulator
from pydrake.systems.framework import DiagramBuilder
from pydrake.visualization import AddDefaultVisualization, ModelVisualizer
import sys
from volta.paths import PROJECT_ROOT
PATH_TO_PROJECT = str(PROJECT_ROOT)

from volta.prelude import *

In [2]:
plant,robot_ind = DefinePlant(model_name="Volta_rev5",
                            is_fixed=True,
                            fixed_frame="torso")
plant_context = plant.CreateDefaultContext()

In [3]:
print(plant.CalcCenterOfMassPositionInWorld(plant_context))
L_foot_contact = plant.GetFrameByName("L_foot_contact") 
W_T_L = L_foot_contact.CalcPoseInWorld(plant_context)
plant.CalcCenterOfMassPositionInWorld(plant_context) - W_T_L.translation()

[-0.00486555 -0.00020195 -0.18470657]


array([-0.00486248, -0.12520312,  0.42129309])

In [4]:
from pydrake.multibody.inverse_kinematics import InverseKinematics, ComPositionConstraint
from pydrake.multibody.plant import MultibodyPlant
from pydrake.solvers import Solve
import numpy as np    
from numpy import ndarray        
def SolveIKWithComPositionConstraint(model_series:str, 
                                    plant:MultibodyPlant, 
                                    is_L_stance:bool,
                                    st_P_com:ndarray,
                                    st_R_base:RotationMatrix,
                                    st_P_sw:ndarray,
                                    st_R_sw:RotationMatrix,
                                    q_initial_guess:list=None):
    IK_solver = InverseKinematics(plant)

    if is_L_stance:
        stance_leg = plant.GetFrameByName("L_foot_contact")
        swing_leg = plant.GetFrameByName("R_foot_contact")
    else:
        stance_leg = plant.GetFrameByName("R_foot_contact")
        swing_leg = plant.GetFrameByName("L_foot_contact")
    base_frame = plant.GetFrameByName("torso")

    # Add position and orientation constraints
    IK_solver.AddPositionConstraint(
        frameB=swing_leg,
        p_BQ=np.zeros(3),     # point in end-effector frame
        frameA=stance_leg,
        p_AQ_lower=st_P_sw-0.0001,
        p_AQ_upper=st_P_sw+0.0001
    )
    IK_solver.AddOrientationConstraint(
        frameAbar=base_frame,
        R_AbarA=st_R_base,
        frameBbar=stance_leg,
        R_BbarB=st_R_base,
        theta_bound=0.0001  # tolerance in radians
    )
    IK_solver.AddOrientationConstraint(
        frameAbar=stance_leg,
        R_AbarA=st_R_sw,
        frameBbar=swing_leg,
        R_BbarB=st_R_sw,
        theta_bound=0.0001  # tolerance in radians
    )
    q_variables = IK_solver.q()
   
    if model_series == "Hanuman":
        joint_limits = [(-3.1416, 3.1416), 
                        (-0.5236, 0.5236), 
                        (-1.5708, 2.0944), 
                        (-2.2689, 0.0000), 
                        (-1.7453, 1.3963), 
                        (-0.3491, 0.3491),
                        (-3.1416, 3.1416), 
                        (-0.5236, 0.5236), 
                        (-1.5708, 2.0944), 
                        (-2.2689, 0.0000), 
                        (-1.7453, 1.3963), 
                        (-0.3491, 0.3491)]
        
    elif model_series == "Volta":
        joint_limits = [(-1.5708, 1.5708), 
                        (-0.6, 0.6), 
                        (-0.7854, 0.7854), 
                        (-2.35, 0.0), 
                        (-1.0, 1.0), 
                        (-1.5708, 1.5708),

                        (-1.5708, 1.5708), 
                        (-0.6, 0.6), 
                        (-0.7854, 0.7854),  
                        (0.0, 2.35), 
                        (-1.0, 1.0),
                        (-1.5708, 1.5708)
                        ]
    
    for i in range(plant.num_positions()):
        IK_solver.prog().AddBoundingBoxConstraint(joint_limits[i][0], joint_limits[i][1], q_variables[i])
    # Create a ComPositionConstraint
    # This constrains the center of mass position relative to a reference frame
    com_constraint = ComPositionConstraint(plant=plant,
                                          model_instances=None,
                                          expressed_frame=stance_leg,
                                          plant_context=plant.CreateDefaultContext())
    # Create a decision variable for the desired center of mass position
    # This is p_EC in the constraint equation: p_EScm(q) - p_EC = 0
    desired_com_position = IK_solver.prog().NewContinuousVariables(3, "com_position")
    # Add the constraint to the optimization program
    # This enforces that the actual CoM position equals the desired position
    IK_solver.prog().AddConstraint(
        constraint=com_constraint,
        vars=np.concatenate([IK_solver.q(), desired_com_position])
    )
    # Set the desired center of mass position 
    IK_solver.prog().AddLinearEqualityConstraint(
        Aeq=np.eye(3),
        beq=st_P_com,
        vars=desired_com_position
    )
    if q_initial_guess is not None:
        q_initial_guess = q_initial_guess
    else:
        q_initial_guess = np.ones(plant.num_positions())*0.1
    plant.SetPositions(context=plant.CreateDefaultContext(),
                       q=q_initial_guess)
    P_W_com = plant.CalcCenterOfMassPositionInWorld(context=plant.CreateDefaultContext())
    T_W_st = stance_leg.CalcPoseInWorld(context=plant.CreateDefaultContext())
    P_st_com = T_W_st.inverse() @ P_W_com
    initial_guess = np.concatenate((q_initial_guess,P_st_com)) 
    result = Solve(IK_solver.prog(),initial_guess)
    is_success = False
    if result.is_success():
        q_solution = result.GetSolution(q_variables)
        is_success = True
    else:
        q_solution = np.zeros(plant.num_positions())
    threshold = 1.5e-03
    q_solution[abs(q_solution) <= threshold] = 0.0 
    return is_success,q_solution

In [5]:
import time
init_time = time.time()
is_success,q = SolveIKWithComPositionConstraint(model_series="Volta",
                                plant=plant,
                                is_L_stance=True,
                                st_P_com=np.array([0.0,-0.125,0.38]),
                                st_R_base=RotationMatrix.Identity(),
                                st_P_sw=np.array([0.0,-0.250,0.0]),
                                st_R_sw=RotationMatrix.Identity()
)

In [6]:
is_success

True

In [7]:
q

array([-0.47109692,  0.        ,  0.        , -1.04979816, -0.57835165,
        0.        ,  0.47113133,  0.        ,  0.        ,  1.05066636,
        0.57927751,  0.        ])

In [8]:
plant.CalcTotalMass(plant_context)

7.1179999999999986

In [8]:
plant.SetPositions(plant_context,q)
print(plant.CalcCenterOfMassPositionInWorld(plant_context))

[ 1.01457731e-02  1.07908056e-06 -2.14314473e-01]


In [9]:
L_foot_contact = plant.GetFrameByName("L_foot_contact") 
W_T_L = L_foot_contact.CalcPoseInWorld(plant_context)
plant.CalcCenterOfMassPositionInWorld(plant_context) - W_T_L.translation()

array([-1.85255657e-05, -1.25000672e-01,  3.50006433e-01])

In [10]:
# -0.1   0.08 -0.3
torso_x_E = 0.0
torso_y_E = -0.02
torso_z_E = -0.47
is_L_success,left_q = SolveInverseKinematics(model_series="Volta",
                                             plant=plant,
                                             is_left=True,
                                             base_P_e=np.array([torso_x_E, torso_y_E,  torso_z_E]),
                                             base_R_e=RollPitchYaw(0, 0, 0).ToRotationMatrix())
print("L")
print(is_L_success)
print(left_q)

L
True
[-0.49438    -0.32086157 -0.16846472 -1.18356467 -0.71616435 -0.36111323]


In [11]:
torso_x_E = 0.0
torso_y_E = -0.27
torso_z_E = -0.47

is_R_success,right_q = SolveInverseKinematics(model_series="Volta",
                                              plant=plant,
                                              is_left=False,
                                              base_P_e=np.array([torso_x_E, torso_y_E,  torso_z_E]),
                                              base_R_e=RollPitchYaw(0, 0, 0).ToRotationMatrix())
print("R")
print(is_R_success)
print(right_q)

R
True
[ 0.49367933 -0.31879286 -0.16712379  1.1827307   0.71566249 -0.35867534]


In [12]:
plant,robot_ind = DefinePlant(model_name="Volta_rev4",
                            is_fixed=True,
                            fixed_frame="torso")
plant_context = plant.CreateDefaultContext()
plant.SetPositions(plant_context,q)
L_foot_contact = plant.GetFrameByName("L_foot_contact") 
W_T_L = L_foot_contact.CalcPoseInWorld(plant_context)
print("Desire eff positions")
print(np.array([torso_x_E, torso_y_E,  torso_z_E]))
print("Positions L from IK")
print(W_T_L.translation())
R_foot_contact = plant.GetFrameByName("R_foot_contact") 
W_T_R = R_foot_contact.CalcPoseInWorld(plant_context)
print("Positions R from IK")
print(W_T_R.translation())

Desire eff positions
[ 0.   -0.27 -0.47]
Positions L from IK
[ 0.0101643   0.12500175 -0.56432091]
Positions R from IK
[ 0.01007195 -0.12500157 -0.56421053]


In [13]:
q

array([-0.20692667,  0.        ,  0.        , -0.4607278 , -0.25357232,
        0.        ,  0.20757996,  0.        ,  0.        ,  0.4627881 ,
        0.25488333,  0.        ])

In [14]:
plant.CalcCenterOfMassPositionInWorld(plant_context)

array([ 1.01457731e-02,  1.07908056e-06, -2.14314473e-01])

In [15]:
p,ind = DefinePlant(model_name="Volta_rev4",
                            is_fixed=True,
                            fixed_frame="torso")
p.SetPositions(p.CreateDefaultContext(),q)


In [16]:
p.CalcCenterOfMassPositionInWorld(p.CreateDefaultContext())

array([-6.38729004e-03,  1.06933064e-06, -2.19631716e-01])

In [17]:
# Start Meshcat
meshcat = StartMeshcat()
meshcat.SetProperty("/drake/visualizer", "visible", False)

INFO:drake:Meshcat listening for connections at http://localhost:7003


In [18]:
# Create ModelVisualizer and add robot
visualizer = ModelVisualizer(meshcat=meshcat,
                             visualize_frames=True,
                             triad_length=0.05,
                             triad_radius=0.001)
urdf_name = "Volta_rev5.urdf"
robot_urdf_path = f"{PATH_TO_PROJECT}/Models/{urdf_name}"
robot = visualizer.parser().AddModels(robot_urdf_path)
plant_viz = visualizer.parser().plant()
robot_body_frame = plant_viz.GetFrameByName("torso")
init_pos = [0.0,0.0,0.0] 
init_orien = np.asarray([0, 0, 0])
X_floorRobot = RigidTransform(
    RollPitchYaw(init_orien * np.pi / 180), p=[0.0,0.0,0.0])
plant_viz.WeldFrames(plant_viz.world_frame(),robot_body_frame, X_floorRobot)

visualizer.Finalize()

# Create context
context = plant_viz.CreateDefaultContext()
# Double stance
# left_q = np.array([-0.46738324, -0.31741558, -0.15638652, -1.1163273 , -0.66869921,
#        -0.35273243])
# right_q = np.array([ 0.4633971 , -0.31761243, -0.15484457,  1.10613954,  0.66731962,
#        -0.35227575])
# Single stance
left_q = np.array([-0.52674564,  0.00213906,  0.        , -1.3643184 , -0.83733544,
        0.00250483])
right_q = np.array([0.53103344, 0.00227564, 0.        , 1.36433905, 0.83306136,
       0.00264209])

visualizer.Run(position=q,loop_once=False)
# visualizer.Run(loop_once=False)


INFO:drake:Click 'Stop Running' or press Esc to quit


In [ ]:
q

array([-0.57762061,  0.28264221,  0.17997203, -1.15649621, -0.60429868,
        0.3337884 ,  0.57708955,  0.28245385,  0.17965164,  1.15596879,
        0.60419404,  0.3333638 ])

In [ ]:
plant_temp,dump = DefinePlant(model_name="Volta_rev4",
                            is_fixed=True,
                            fixed_frame="torso")
p = CalcForwardKinematics(plant=plant_temp,
                            joint_angles=np.concatenate((left_q, right_q)),
                            base_frame_name="torso",
                            end_effector_frame_name="L_foot_contact")
print(p)

In [ ]:
plant.CalcTotalMass(plant.CreateDefaultContext(),robot_ind)

In [ ]:
temp_p,robot_ind = DefinePlant(model_name="Volta_rev4_only_R",
                            is_fixed=True,
                            fixed_frame="R_foot_contact")
# temp_p.SetPositions(temp_p.CreateDefaultContext(),-left_q[::-1])
p = temp_p.CalcCenterOfMassPositionInWorld(temp_p.CreateDefaultContext(),robot_ind)
print("Center of mass position in world frame")
print(p)

In [ ]:
temp_p.CalcTotalMass(temp_p.CreateDefaultContext(),robot_ind)

In [ ]:
temp_p,robot_ind = DefinePlant(model_name="Volta_rev4_L_fixed",
                            is_fixed=True,
                            fixed_frame="L_foot_contact")
temp_p.CalcTotalMass(temp_p.CreateDefaultContext(),robot_ind)